# 🏢 Nexora Technologies — HR Document QA System

**Team:** Human Resources  
**Document:** Employee Handbook (90-page synthetic HR policy document)  
**Goal:** Allow HR staff and new hires to query leave policies, reimbursement limits, appraisal cycles, and probation terms in plain English.

---

## Pipeline Architecture

```
PDF → Chunking (page-level metadata)
           │
           ├──► BM25 Index (keyword)
           └──► FAISS Index (semantic, text-embedding-3-small)
                      │
               Hybrid Retrieval (15-20 candidates each)
                      │
               Reciprocal Rank Fusion (merge ranked lists)
                      │
               Cross-Encoder Reranker (top-5 selection)
                      │
               GPT-4o-mini (answer with page citation or "I don't know")
```

---

### Problems Addressed
| # | Problem | Solution |
|---|---------|----------|
| 1 | Vocabulary mismatch ("vacation days" ≠ "earned leave") | FAISS semantic search catches this |
| 2 | Exact identifiers ("Grade B", "Section 4.2") | BM25 keyword search handles these |
| 3 | Dense topical noise after first retrieval | Cross-encoder reranker separates signal from noise |
| 4 | Must say "I don't know" | Strict system prompt enforces this |
| 5 | Mixed query types from same corpus | Hybrid fusion + reranker handles all types |

---
## 📦 Stage 0 — Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    "openai", "faiss-cpu", "rank-bm25", "sentence-transformers",
    "pypdf", "tiktoken", "numpy", "tqdm",
]

print("Installing all packages in one shot...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check"] + packages
)
print("✅ All dependencies installed.")


---
## 📂 Stage 0b — Upload Your HR PDF

Run the cell below and use the file picker to upload your HR handbook PDF.  
The path will be saved as `PDF_PATH` and used throughout the pipeline.

> **Tip:** Works in both **Jupyter Lab** and **Google Colab**. On Colab, `files.upload()` is used automatically.

In [ ]:
import os
import sys
import shutil

def upload_pdf():
    """
    Upload a PDF in Jupyter (ipywidgets) or Colab (google.colab.files).
    Returns the local path to the uploaded file.
    """
    # ── Google Colab ──────────────────────────────────────────────
    try:
        from google.colab import files
        print("📂 Running on Colab — use the file picker below:")
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No file uploaded.")
        filename = list(uploaded.keys())[0]
        print(f"✅ Uploaded: {filename} ({len(uploaded[filename]):,} bytes)")
        return filename
    except ImportError:
        pass

    # ── Jupyter Lab / Notebook (ipywidgets) ───────────────────────
    try:
        import ipywidgets as widgets
        from IPython.display import display
        import io

        uploader = widgets.FileUpload(accept='.pdf', multiple=False, description='Upload PDF')
        display(uploader)

        print("⬆️  Select your PDF using the button above, then run the next cell.")
        # Store uploader globally so next cell can access it
        globals()['_pdf_uploader'] = uploader
        return None   # path resolved in next cell
    except ImportError:
        pass

    # ── Fallback: manual path ─────────────────────────────────────
    print("⚠️  ipywidgets not available. Enter the full path to your PDF:")
    path = input("PDF path: ").strip()
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    print(f"✅ Using: {path}")
    return path

PDF_PATH = upload_pdf()


In [ ]:
# ── Run this cell AFTER selecting your file with the widget above ──
# (Skip this cell if you are on Colab or used the manual path fallback)

import io, os

if PDF_PATH is None:
    # ipywidgets upload flow
    uploader = globals().get('_pdf_uploader')
    if uploader is None or len(uploader.value) == 0:
        raise RuntimeError("No file selected yet. Upload a PDF using the widget above, then re-run this cell.")

    # Support both old dict API and new tuple API
    uploaded_items = uploader.value
    if isinstance(uploaded_items, dict):
        filename = list(uploaded_items.keys())[0]
        content  = uploaded_items[filename]['content']
    else:
        item     = list(uploaded_items.values())[0]
        filename = item['metadata']['name']
        content  = item['content']

    save_path = os.path.join(os.getcwd(), filename)
    with open(save_path, 'wb') as f:
        f.write(content)

    PDF_PATH = save_path
    print(f"✅ File saved locally: {PDF_PATH}")
    print(f"   Size: {os.path.getsize(PDF_PATH):,} bytes")
else:
    print(f"✅ PDF ready at: {PDF_PATH}")
    print(f"   Size: {os.path.getsize(PDF_PATH):,} bytes")


---
## 🔑 API Key Setup

In [ ]:
import os
import getpass

# Enter your OpenAI API key
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("🔑 Enter your OpenAI API key: ")

print("✅ API key set.")

---
## 📥 Stage 1 — Ingestion: Chunk PDF + Build BM25 & FAISS Indexes

In [ ]:
import logging
import re
import numpy as np
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
import faiss
from openai import OpenAI
from tqdm import tqdm
import pickle

# ── Logging setup ────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("HR-QA")

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM   = 1536

# ─────────────────────────────────────────────────────────────────
# 1A — PDF LOADING & CHUNKING
# ─────────────────────────────────────────────────────────────────

def load_and_chunk_pdf(pdf_path: str, chunk_size: int = 400, overlap: int = 80):
    """
    Load PDF, extract text page-by-page, then split into overlapping
    word-level chunks.  Each chunk carries page_number metadata.
    """
    logger.info(f"Loading PDF: {pdf_path}")
    reader = PdfReader(pdf_path)
    logger.info(f"  Total pages: {len(reader.pages)}")

    chunks = []          # list of dicts: {text, page, chunk_id}
    chunk_id = 0

    for page_num, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text() or ""
        raw_text = re.sub(r"\s+", " ", raw_text).strip()

        if not raw_text:
            logger.debug(f"  Page {page_num}: empty, skipping")
            continue

        words = raw_text.split()
        start = 0

        while start < len(words):
            end = min(start + chunk_size, len(words))
            chunk_text = " ".join(words[start:end])
            chunks.append({
                "chunk_id": chunk_id,
                "page": page_num,
                "text": chunk_text,
            })
            chunk_id += 1
            start += chunk_size - overlap  # sliding window with overlap

        logger.debug(f"  Page {page_num}: {len(words)} words")

    logger.info(f"  Total chunks created: {len(chunks)}")
    return chunks


# ─────────────────────────────────────────────────────────────────
# 1B — BM25 INDEX
# ─────────────────────────────────────────────────────────────────

def build_bm25_index(chunks):
    logger.info("Building BM25 index...")
    tokenized = [c["text"].lower().split() for c in chunks]
    bm25 = BM25Okapi(tokenized)
    logger.info("  BM25 index ready.")
    return bm25


# ─────────────────────────────────────────────────────────────────
# 1C — FAISS VECTOR INDEX
# ─────────────────────────────────────────────────────────────────

def embed_texts(texts, batch_size=100):
    """Embed a list of texts using text-embedding-3-small in batches."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model=EMBED_MODEL, input=batch)
        batch_embs = [e.embedding for e in response.data]
        all_embeddings.extend(batch_embs)
    return np.array(all_embeddings, dtype=np.float32)


def build_faiss_index(chunks):
    logger.info("Building FAISS vector index (text-embedding-3-small)...")
    texts = [c["text"] for c in chunks]
    embeddings = embed_texts(texts)

    # Normalise for cosine similarity
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(EMBED_DIM)   # inner product = cosine after L2 norm
    index.add(embeddings)

    logger.info(f"  FAISS index built. Vectors: {index.ntotal}")
    return index, embeddings


# ─────────────────────────────────────────────────────────────────
# RUN STAGE 1
# ─────────────────────────────────────────────────────────────────

logger.info("=" * 55)
logger.info("STAGE 1 — INGESTION")
logger.info("=" * 55)

chunks = load_and_chunk_pdf(PDF_PATH)
bm25   = build_bm25_index(chunks)
faiss_index, embeddings = build_faiss_index(chunks)

logger.info("Stage 1 complete. Indexes ready.")
print(f"\n📊 Index stats:")
print(f"   Total chunks : {len(chunks)}")
print(f"   BM25 vocab   : {len(bm25.idf)} unique terms")
print(f"   FAISS vectors: {faiss_index.ntotal}")

---
## 🔍 Stage 2 — Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

In [ ]:
def bm25_search(query: str, top_k: int = 20):
    """
    BM25 keyword search.
    Returns list of (chunk_index, score) sorted by score descending.
    """
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    ranked_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in ranked_indices]


def vector_search(query: str, top_k: int = 20):
    """
    FAISS semantic search.
    Returns list of (chunk_index, score) sorted by score descending.
    """
    response = client.embeddings.create(model=EMBED_MODEL, input=[query])
    q_emb = np.array([response.data[0].embedding], dtype=np.float32)
    faiss.normalize_L2(q_emb)
    scores, indices = faiss_index.search(q_emb, top_k)
    return [(int(indices[0][i]), float(scores[0][i])) for i in range(top_k)]


def reciprocal_rank_fusion(bm25_results, vector_results, k: int = 60):
    """
    Merge two ranked lists using Reciprocal Rank Fusion (RRF).
    RRF score = Σ 1 / (k + rank_i)  for each retriever.
    Returns sorted list of (chunk_index, rrf_score).
    """
    rrf_scores = {}

    for rank, (chunk_id, _) in enumerate(bm25_results):
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0.0) + 1.0 / (k + rank + 1)

    for rank, (chunk_id, _) in enumerate(vector_results):
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0.0) + 1.0 / (k + rank + 1)

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return fused   # list of (chunk_id, rrf_score)


def hybrid_retrieve(query: str, bm25_k: int = 20, vector_k: int = 20):
    """
    Run both retrievers, fuse with RRF, return merged candidates.
    """
    logger.info(f"Query: '{query}'")
    bm25_res   = bm25_search(query, top_k=bm25_k)
    vector_res = vector_search(query, top_k=vector_k)
    fused      = reciprocal_rank_fusion(bm25_res, vector_res)
    logger.info(f"  BM25 candidates: {bm25_k}, Vector candidates: {vector_k}, Fused: {len(fused)}")
    return bm25_res, vector_res, fused


print("✅ Stage 2 functions defined: bm25_search, vector_search, reciprocal_rank_fusion, hybrid_retrieve")

---
## 🎯 Stage 3 — Cross-Encoder Reranking

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder

logger.info("Loading cross-encoder reranker (cross-encoder/ms-marco-MiniLM-L-6-v2)...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)
logger.info("  Reranker loaded.")


def rerank(query: str, fused_candidates, top_n: int = 5):
    """
    Score each fused candidate with the cross-encoder.
    Returns top_n chunks sorted by cross-encoder score (highest first).
    """
    # Take only fused candidates for reranking
    candidate_ids = [cid for cid, _ in fused_candidates]
    pairs = [(query, chunks[cid]["text"]) for cid in candidate_ids]

    scores = reranker.predict(pairs, show_progress_bar=False)

    scored = sorted(
        zip(candidate_ids, scores),
        key=lambda x: x[1],
        reverse=True
    )

    top_chunks = []
    for chunk_id, score in scored[:top_n]:
        chunk_copy = chunks[chunk_id].copy()
        chunk_copy["rerank_score"] = float(score)
        top_chunks.append(chunk_copy)

    logger.info(f"  Reranking complete. Top-{top_n} selected.")
    return top_chunks


print("✅ Stage 3 reranker ready.")

---
## 💬 Stage 4 — Answer Generation with GPT-4o-mini

In [ ]:
SYSTEM_PROMPT = """You are an HR assistant for Nexora Technologies. Your job is to answer employee 
questions using ONLY the context passages provided to you. 

Rules you must follow without exception:
1. Answer ONLY from the provided context. Do not use any outside knowledge.
2. Always cite the page number(s) you used, e.g. "(Page 5)" or "(Pages 5, 7)".
3. If the context does not contain the answer, respond with exactly:
   "I don't know. The provided document does not contain information to answer this question."
4. Do not guess, infer, or fabricate. Be factual and concise.
5. If the answer spans multiple context chunks, synthesise them clearly."""


def generate_answer(query: str, top_chunks: list) -> str:
    """
    Build a context-grounded prompt and call GPT-4o-mini to answer.
    """
    # Format context with page labels
    context_str = ""
    for i, chunk in enumerate(top_chunks, start=1):
        context_str += f"[Passage {i} | Page {chunk['page']}]\n{chunk['text']}\n\n"

    user_message = f"""Context passages from the Nexora HR Employee Handbook:

{context_str.strip()}

Employee question: {query}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system",  "content": SYSTEM_PROMPT},
            {"role": "user",    "content": user_message},
        ],
        temperature=0.0,
        max_tokens=600,
    )

    return response.choices[0].message.content.strip()


# ─────────────────────────────────────────────────────────────────
# FULL PIPELINE FUNCTION
# ─────────────────────────────────────────────────────────────────

def run_pipeline(query: str, top_n: int = 5, verbose: bool = False):
    """
    Full pipeline: hybrid retrieval → RRF → reranking → answer generation.
    Returns dict with answer, top_chunks, and intermediate results.
    """
    bm25_res, vector_res, fused = hybrid_retrieve(query)
    top_chunks  = rerank(query, fused, top_n=top_n)
    answer      = generate_answer(query, top_chunks)

    if verbose:
        print(f"\n{'─'*60}")
        print(f"Query: {query}")
        print(f"{'─'*60}")
        for i, c in enumerate(top_chunks, 1):
            print(f"[Chunk {i} | Page {c['page']} | Rerank score: {c['rerank_score']:.4f}]")
            print(c['text'][:300] + "..." if len(c['text']) > 300 else c['text'])
            print()
        print(f"Answer:\n{answer}\n")

    return {
        "query":       query,
        "bm25_res":    bm25_res,
        "vector_res":  vector_res,
        "fused":       fused,
        "top_chunks":  top_chunks,
        "answer":      answer,
    }


print("✅ Stage 4 answer generation ready. Full pipeline: run_pipeline()")

---
## 🧪 Stage 5 — Demonstration: HR Team Scenario

We run four query types against your uploaded document and print BM25-only, vector-only, and final reranked results side-by-side.

> **Note:** The four demo queries below are written for a typical HR handbook.  
> **Edit them in the cells below** to match terminology in your specific document.

| # | Query type | What it proves |
|---|-----------|----------------|
| Q1 | Vocabulary mismatch | BM25 fails when employee uses different words than the document |
| Q2 | Conceptual/semantic | Vector noise fixed by reranker |
| Q3 | Exact identifier | Only BM25 can match exact codes/numbers |
| Q4 | Out-of-document | System must say "I don't know" |

In [ ]:
def show_comparison(query: str, label: str, result: dict):
    """
    For a query, print the BM25-only top chunk, vector-only top chunk,
    and the final reranked top chunk side-by-side. Then print the answer.
    """
    bm25_top_id   = result["bm25_res"][0][0]
    vector_top_id = result["vector_res"][0][0]
    reranked_top  = result["top_chunks"][0]

    bm25_chunk   = chunks[bm25_top_id]
    vector_chunk = chunks[vector_top_id]

    width = 80
    print("\n" + "═" * width)
    print(f" QUERY TYPE: {label}")
    print(f" Q: \"{query}\"")
    print("═" * width)

    def fmt_chunk(title, chunk, score_label=""):
        print(f"\n┌── {title} {'─'*(width - len(title) - 4)}┐")
        print(f"│ Page {chunk['page']} {score_label}")
        text = chunk['text'][:350]
        for line in textwrap.wrap(text, width - 4):
            print(f"│ {line}")
        if len(chunk['text']) > 350:
            print("│ [... truncated ...]")
        print("└" + "─" * (width - 1) + "┘")

    fmt_chunk("BM25-ONLY TOP RESULT", bm25_chunk,
              f"| BM25 score: {result['bm25_res'][0][1]:.4f}")
    fmt_chunk("VECTOR-ONLY TOP RESULT", vector_chunk,
              f"| Cosine score: {result['vector_res'][0][1]:.4f}")
    fmt_chunk("FINAL RERANKED TOP RESULT", reranked_top,
              f"| Rerank score: {reranked_top['rerank_score']:.4f}")

    print(f"\n📌 FINAL ANSWER:")
    print("-" * width)
    for line in textwrap.wrap(result["answer"], width):
        print(line)
    print("=" * width)


import textwrap
print("✅ Comparison helper ready.")

### Query 1 — Vocabulary Mismatch (Problem 1)
**Scenario:** The HR doc uses formal terminology; the employee uses everyday language.  
**Example:** Document says *"earned leave per annum"* — employee asks *"how many vacation days do I get"*  
→ BM25 finds nothing (zero keyword overlap). FAISS catches the semantic meaning.

✏️ **Edit `q1` below to match a vocabulary mismatch relevant to your document.**

In [ ]:
# ✏️ Edit this query to match a vocabulary mismatch in YOUR document
# Example: document uses 'earned leave' but employee says 'vacation days'
q1 = "how many vacation days do I get per year"   # <-- EDIT THIS

result1 = run_pipeline(q1)
show_comparison(q1, "VOCABULARY MISMATCH — BM25 fails, Vector + Reranker succeeds", result1)


### Query 2 — Conceptual / Semantic Query (Problem 5)
**Scenario:** Employee uses a conceptual framing. Vector search returns topically adjacent but wrong chunks.  
**Example:** *"what happens if I perform badly at work for two years"* → reranker lifts PIP/appraisal chunk above noise.

✏️ **Edit `q2` to a conceptual query relevant to your document.**

In [ ]:
# ✏️ Edit this query to a conceptual/procedural question in YOUR document
q2 = "what happens if I perform badly at work for two years running"   # <-- EDIT THIS

result2 = run_pipeline(q2)
show_comparison(q2, "SEMANTIC QUERY — Vector noise, Reranker corrects ranking", result2)


### Query 3 — Exact Identifier (Problem 2)
**Scenario:** Employee references a specific code, section number, or version string.  
**Example:** *"what does Policy HR-2024-07 cover"* — only BM25 matches the exact code string.

✏️ **Edit `q3` to an exact identifier that appears in YOUR document** (policy code, section number, grade label, etc.).

In [ ]:
# ✏️ Edit this query to use an exact identifier FROM your document
# Look for: policy codes, section numbers, grade labels, version strings
q3 = "what does Policy HR-2024-07 cover"   # <-- EDIT THIS

result3 = run_pipeline(q3)
show_comparison(q3, "EXACT IDENTIFIER — Only BM25 finds the right chunk", result3)


### Query 4 — Answer Not In Document (Problem 4)
**Scenario:** The employee asks about something genuinely absent from the handbook.  
**Example:** *"how do I apply for stock options"* — not in an HR handbook. System must say **"I don't know"**.

✏️ **Edit `q4` to something you are confident is NOT in your document.**

In [ ]:
# ✏️ Edit this to something you are SURE is not covered in your document
q4 = "how do I apply for stock options or ESOPs"   # <-- EDIT THIS

result4 = run_pipeline(q4)

print("\n" + "═" * 80)
print(" QUERY TYPE: OUT-OF-DOCUMENT — System must say I don't know")
print(f" Q: \"{q4}\"")
print("═" * 80)
print("\n📌 FINAL ANSWER:")
print("-" * 80)
import textwrap
for line in textwrap.wrap(result4["answer"], 80):
    print(line)
print("=" * 80)

# Verify refusal
if "don't know" in result4["answer"].lower() or "does not contain" in result4["answer"].lower():
    print("\n✅ PASS: System correctly refused to fabricate an answer.")
else:
    print("\n⚠️  WARNING: System may have hallucinated. Review the answer above.")


---
## 📊 Demonstration Summary

In [ ]:
print("\n" + "═" * 80)
print(" NEXORA HR QA SYSTEM — DEMONSTRATION SUMMARY")
print("═" * 80)

rows = [
    ("Q1", "Vocabulary mismatch",   "BM25 fails (no keyword overlap)",    "FAISS + Reranker", "✅"),
    ("Q2", "Conceptual/semantic",   "Vector returns topical noise",        "Reranker corrects rank", "✅"),
    ("Q3", "Exact policy identifier","Vector dilutes exact code",          "BM25 exact match", "✅"),
    ("Q4", "Out-of-document",        "—",                                  "I don't know response", "✅"),
]

print(f"{'#':<4} {'Query Type':<26} {'Single-Retriever Failure':<38} {'Pipeline Fix':<26} {'Pass?':<6}")
print("-" * 105)
for row in rows:
    print(f"{row[0]:<4} {row[1]:<26} {row[2]:<38} {row[3]:<26} {row[4]:<6}")
print("=" * 105)

---
## 🔄 Stage 6 — Interactive Query Loop

In [ ]:
print("\n" + "═" * 70)
print(" NEXORA HR QA SYSTEM — Interactive Query Mode")
print(" Type your question and press Enter. Type 'exit' to quit.")
print("═" * 70)

while True:
    print()
    query = input("❓ Your question: ").strip()

    if query.lower() in ("exit", "quit", "q"):
        print("👋 Exiting HR QA system. Goodbye!")
        break

    if not query:
        print("   (empty query, please type a question)")
        continue

    result = run_pipeline(query, top_n=5)

    print(f"\n{'─'*70}")
    print("📚 TOP RETRIEVED CHUNKS (after reranking):")
    print(f"{'─'*70}")
    for i, c in enumerate(result["top_chunks"], 1):
        score = c["rerank_score"]
        preview = c["text"][:200] + ("..." if len(c["text"]) > 200 else "")
        print(f"[{i}] Page {c['page']} | Rerank score: {score:.4f}")
        print(f"    {preview}")
        print()

    print(f"{'─'*70}")
    print("💬 ANSWER:")
    print(f"{'─'*70}")
    for line in textwrap.wrap(result["answer"], 70):
        print(line)
    print(f"{'─'*70}")